[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pytorch/lab-p4-torch-on-tpu.ipynb)

# LAB·P4 · Torch on a TPU

**Hardware:** Colab TPU. Set Runtime → Change runtime type → TPU before anything else, and restart the runtime if you installed a different accelerator stack earlier in the session. Every code cell in this lab is Colab-TPU-only; none of it runs on a CPU runtime.

Chapter 10 named two bridges from torch to the TPU: torchax, which backs a `torch.Tensor` with a `jax.Array` and runs your module as JAX underneath, and torch_xla, which records ops on lazy tensors and materializes them at a step boundary. This lab runs torchax through its documented surface (eager forward, jit, checkpoints), runs the chapter 4 training loop through torch_xla, and closes with the capstone: a full-state checkpoint, a simulated kill, a resume, and proof the resumed run picks the loss curve back up where it left off.

One rule before you start: **the two bridges cannot share a runtime.** Each installs its own interception into torch process-wide, so this lab runs torchax first, restarts the runtime, and then runs torch_xla. The restart cell says so when you reach it.

Before every reveal cell there is an empty "your prediction" cell above it. Write your answer there, then run the reveal and compare.


## Bridge one: torchax

torchax is a `torch.Tensor` subclass that holds a `jax.Array` and intercepts PyTorch's dispatcher: once enabled, an op on a torchax tensor looks like ordinary torch and runs as JAX. The install is three packages: a CPU torch wheel (torchax needs torch's Python surface, not its CUDA kernels), `jax[tpu]` for the accelerator, and torchax itself.

In [ ]:
# Colab TPU runtime only
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q -U 'jax[tpu]'
!pip install -q torchax

import jax
print(jax.__version__, jax.devices())

## Enable torchax, move a model to the jax device

`torchax.enable_globally()` turns the interception on. From there, `.to('jax')` on a tensor or a module is the whole change: the same module code now runs its ops as JAX.

**your prediction:** after moving both the model and its input to `'jax'`, what type does calling the model return?

In [ ]:
# Colab TPU runtime only
import torch
import torch.nn as nn
import torchax

torchax.enable_globally()

class Mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.up = nn.Linear(8, 32)
        self.down = nn.Linear(32, 1)

    def forward(self, x):
        return self.down(torch.relu(self.up(x)))

model = Mlp().to('jax')
inputs = torch.randn(4, 8, device='jax')
out = model(inputs)
print(type(out))     # torchax.tensor.Tensor
print(out.jax().shape)   # the underlying jax.Array

`out` comes back as `torchax.tensor.Tensor`, a wrapper you can mostly ignore: it behaves like a `torch.Tensor` in every way that matters, and `.jax()` unwraps the `jax.Array` underneath when you need to hand it to JAX-native code directly.

## Forward and jit, the documented torchax surface

torchax documents three things today: the eager forward you just ran, jit through `JittableModule` (or `functional_call` plus `jax_jit`), and the checkpoint helpers. Training with plain `loss.backward()` and `torch.optim` is not on that list, and running it fails inside torch's autograd: the documented training route is the functional one (parameters as inputs, gradients from jax, optax as the optimizer). This lab keeps torchax to its documented surface and trains over the other bridge below, where the eager loop is the documented path.

**your prediction:** wrapping the model in `JittableModule` and calling it twice with the same shape: how many compilations?


In [ ]:
# Colab TPU runtime only
from torchax.interop import JittableModule

m_jitted = JittableModule(model)
out1 = m_jitted(inputs)   # first call: traces and compiles
out2 = m_jitted(inputs)   # same shapes: the compiled program is reused
print(type(out1), out1.jax().shape)


## Restart the runtime before bridge two

`torchax.enable_globally()` is exactly what it says: it installs torch modes for the whole process, so every torch op routes through torchax until something takes them down. torch_xla claims the same territory. Run them in one kernel and the tensors torch_xla hands to autograd no longer carry a graph, so `loss.backward()` raises `element 0 of tensors does not require grad and does not have a grad_fn`. torchax ships `torchax.disable_globally()` for the polite version of this, and a fresh process is the guarantee. The `torch_xla` install below wants a restart anyway, since torch is already imported by now.

**Do this now: Runtime → Restart session.** Then run the cells below, which are self-contained: they import everything they need and never touch torchax.

## Bridge two: torch_xla

torch_xla takes the other route: ops on an `'xla'` tensor record instead of executing, and a step boundary (`torch_xla.step()` per iteration, `torch_xla.sync()` to flush) materializes the recorded graph through XLA. Nothing runs until the boundary asks for it. This is the bridge where the chapter 4 loop runs unchanged, `loss.backward()` and `opt.step()` included: eager-style training is the path torch_xla documents.

**your prediction:** compared to the torchax forward above, what changes in the training loop's shape, not just its device string?

In [ ]:
# Colab TPU runtime only · run AFTER restarting the session
# torch and torch_xla ship as a matched pair: the two versions must agree.
# Check github.com/pytorch/xla for the current pair if this one has aged out.
!pip install -q torch==2.8.0 'torch_xla[tpu]==2.8.0'

In [ ]:
# Colab TPU runtime only · self-contained after the restart
import torch
import torch.nn as nn
import torch_xla

torch.manual_seed(0)
x = torch.randn(256, 8).to('xla')
y = x.sum(dim=1, keepdim=True)
model = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 1)).to('xla')
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for step in range(200):
    with torch_xla.step():
        loss = nn.functional.mse_loss(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
torch_xla.sync()
print(f"final loss {loss.item():.5f}")

## Reading the sync-point model

The `with torch_xla.step():` block is the unit torch_xla schedules: everything inside it records against lazy tensors, and `torch_xla.sync()` is where the recorded graph actually runs. Change a shape between steps and the boundary recompiles, the classic torch_xla cost the blueprint names in chapter 10. torchax has no equivalent boundary in its eager path: every op runs as JAX immediately, and the compilation boundary only appears if you opt into `jax_jit` yourself. Same destination, StableHLO underneath both, two different places where the framework hands control to the compiler.

## Capstone: full-state checkpoint, kill, resume, prove the overlay

The bar from chapter 12: train, checkpoint the whole state (model, optimizer, step), kill the run, resume from the checkpoint, and prove the resumed loss curve is the same curve, not a new one. The capstone runs over the torch_xla bridge, and the checkpoint uses `xm.save`, which moves device tensors to host before writing, so the chapter 4 contract (`model.state_dict()`, `opt.state_dict()`, `step`) carries over unchanged. (torchax has its own `save_checkpoint`/`load_checkpoint` pair for the functional world; the ledger it saves is optax state, matching its documented training route.)


In [ ]:
# Colab TPU runtime only · same restarted session as the loop above
import torch
import torch.nn as nn
import torch_xla
import torch_xla.core.xla_model as xm

torch.manual_seed(0)
x = torch.randn(256, 8).to('xla')
y = x.sum(dim=1, keepdim=True)

model = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 1)).to('xla')
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

losses_before_kill = []
for step in range(100):
    with torch_xla.step():
        loss = nn.functional.mse_loss(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    losses_before_kill.append(loss.item())

xm.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step}, "/tmp/ckpt.pt")
print(f"checkpointed at step {step}, loss {losses_before_kill[-1]:.5f}")

# the kill: drop every reference the run above built
del model, opt

# the resume: fresh objects, then the checkpoint overwrites their state
model = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 1)).to('xla')
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loaded = torch.load("/tmp/ckpt.pt")
model.load_state_dict(loaded["model"])
opt.load_state_dict(loaded["opt"])
start_step = loaded["step"] + 1

losses_after_resume = []
for step in range(start_step, start_step + 100):
    with torch_xla.step():
        loss = nn.functional.mse_loss(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    losses_after_resume.append(loss.item())
print(f"resumed from step {start_step}, first loss {losses_after_resume[0]:.5f}")


## Prove the resume

Plot `losses_before_kill + losses_after_resume` as one line and overlay a fresh 200-step run of the same seed that never got killed. The resumed curve should sit on top of the uninterrupted one, not jump or restart from a higher loss: that overlay is the proof, not the checkpoint file existing. Write down the loss at the exact step where the kill happened and the loss the resumed run reports at that same step; they should match to the precision the optimizer's own state reproduces.

## Paste-back: your measured numbers

Fill in the blob below from your own run, then paste it into your notes or the chapter discussion.

```
chip:
dtype:
bridge (torchax / torch_xla):
shapes (batch, features):
loss at kill step:
loss at matching resumed step:
overlay proof (describe or link a plot):
```

In [ ]:
# Colab TPU runtime only
import json
import torch_xla

print(json.dumps({
 "lab": "p4",
 "chip": torch_xla.device_type() if hasattr(torch_xla, "device_type") else str(xm.xla_device()),
 "bridge": "torch_xla",
 "kill_step": step,
 "loss_at_kill": losses_before_kill[-1],
 "loss_at_resume_start": losses_after_resume[0],
}, indent=1))

## Mark it run

Read the three chapters this lab drills: [kernels.rudrite.com/pytorch/bridges](https://kernels.rudrite.com/pytorch/bridges), [kernels.rudrite.com/pytorch/tpu-practice](https://kernels.rudrite.com/pytorch/tpu-practice), and [kernels.rudrite.com/pytorch/training-run](https://kernels.rudrite.com/pytorch/training-run).